# Audio transcriber and summarizer

1. Diarize conversation
2. Transcribe the audio
3. Recognize content type - meeting, speech or conversation
3. Summarize the conversation

If the conversation is
- A meeting: produce minutes of the meeting
- A speech: prodce the speech summar
- A conversation: produce speech summary with speaker labels

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6
!pip install pyannote.audio
!pip install soundfile

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.7/893.7 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 17.2 MB/s eta 0:00:

ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
^C


In [1]:
# imports

import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig, pipeline
import torch
import gradio as gr
from pyannote.audio import Pipeline
import torchaudio
import soundfile as sf


In [2]:
# Sign in to HuggingFace Hub

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

## Step1: Audio Diarization

In [3]:
def audio_diarization(audio_filename, diarization_model= "pyannote/speaker-diarization-3.1", min_speakers=1, max_speakers=5):
    pipeline = Pipeline.from_pretrained(diarization_model)
    pipeline.to(torch.device("cuda"))

    diarization = pipeline(audio_filename, min_speakers=min_speakers, max_speakers=max_speakers)

    return diarization

## Step2: Transcribe Audio

In [4]:
def transcribe(audio_filename, transcription_model= "openai/whisper-medium.en"):
    pipe = pipeline(
        "automatic-speech-recognition",
        model= transcription_model,
        dtype=torch.float16,
        chunk_length_s=30,
        device='cuda',
        return_timestamps=True
    )

    transcription_result = pipe(audio_filename)
    return transcription_result

## step3: Combine Diarization and Transcription

In [5]:
# Extract diarization segments with speaker label

def extract_diarization_segments(diarization):
    diarization_segments = []

    for segment_tuple in diarization.speaker_diarization:
        speech_segment = segment_tuple[0] # The Segment object is the first element of the tuple
        speaker_label = segment_tuple[1] # The speaker label is the second element
        diarization_segments.append({
            'start': speech_segment.start,
            'end': speech_segment.end,
            'speaker': speaker_label
        })

    print(f"Extracted {len(diarization_segments)} diarization segments.")
    #print(diarization_segments[:5]) # Display first 5 segments for verification

    return diarization_segments

In [6]:

def format_transcript(speaker_annotated_transcription):
    formatted_transcript = []

    for chunk in speaker_annotated_transcription:
        speaker = chunk['speaker']
        text = chunk['text']

        if not formatted_transcript or formatted_transcript[-1]['speaker'] != speaker:
            # New speaker or first chunk, create a new entry
            formatted_transcript.append({
                'speaker': speaker,
                'text': text
            })
        else:
            # Same speaker, append text to the last entry
            formatted_transcript[-1]['text'] += text

    markdown_output = ""
    for entry in formatted_transcript:
        markdown_output += f"**{entry['speaker']}**: {entry['text']}\n\n"

    # display(Markdown(markdown_output))
    return markdown_output

In [7]:
# Using simple overlap criteria to label speaker to transcription

def combine_diarization_transcription(diarization_segments, transcription_result):
    speaker_annotated_transcription = []

    for chunk in transcription_result['chunks']:
        chunk_start, chunk_end = chunk['timestamp']
        chunk_text = chunk['text']

        best_overlap = -1
        assigned_speaker = 'Unknown'

        for segment in diarization_segments:
            segment_start = segment['start']
            segment_end = segment['end']
            segment_speaker = segment['speaker']

            # Calculate overlap
            overlap_start = max(chunk_start, segment_start)
            overlap_end = min(chunk_end, segment_end)
            overlap = overlap_end - overlap_start

            if overlap > best_overlap:
                best_overlap = overlap
                assigned_speaker = segment_speaker

        speaker_annotated_transcription.append({
            'start': chunk_start,
            'end': chunk_end,
            'speaker': assigned_speaker,
            'text': chunk_text
        })

    transcription = format_transcript(speaker_annotated_transcription)
    # print(f"Extracted {len(speaker_annotated_transcription)} speaker-annotated chunks.")

    return transcription

## Step4: Summarize

In [24]:

def messages(markdown_output):
    system_message = """
    You are an expert assistant for summarizing audio transcripts. Your task is to analyze the provided transcript,
    which includes speaker annotations, and generate a concise and informative output based on the content type.

    If the content is a general conversation:
    You- Summarize the conversation, clearly detailing who spoke what, using the provided speaker labels.

    If the content is a meeting:
    You- Produce minutes of the meeting in markdown format, including:
    - A summary with attendees, location, and date (if available or implied).
    - Key discussion points, attributing each point to the relevant speaker(s).
    - Key takeaways or decisions made.
    - Action items with owners.

    If the content is a speech:
    You- Summarize the speech with its key points, objectives, goals, and plans, attributing any relevant sections
    to the speaker.

    Always output in markdown format without code blocks.
    """

    user_prompt = f"""
    Here is a speaker-annotated transcript:
    {markdown_output}

    Please categorize the content and provide an appropriate summary or meeting minutes as per your instructions.
    Ensure all speaker contributions are clearly identified.
    """

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt}
    ]
    return messages

In [25]:

def generate_summary(messages, summarization_model= "meta-llama/Llama-2-7b-chat-hf", quantization=True):

    LLAMA = summarization_model
    tokenizer = AutoTokenizer.from_pretrained(LLAMA)
    tokenizer.pad_token = tokenizer.eos_token
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer)
    if quantization:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4"
        )
        model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
    else:
        model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto")
    outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)
    response = tokenizer.decode(outputs[0])

        # Extract only the assistant's summary from the raw LLM output
    assistant_start_marker = "<|start_header_id|>assistant<|end_header_id|>\n\n"
    assistant_end_marker = "<|eot_id|>"

    start_index = response.find(assistant_start_marker)
    end_index = response.rfind(assistant_end_marker)

    if start_index != -1 and end_index != -1:
        # Adjust start_index to point to the beginning of the actual content
        summary = response[start_index + len(assistant_start_marker):end_index].strip()
    else:
        # Fallback if markers are not found, or handle as an error
        summary = "Could not extract summary from LLM output. Raw output: " + response

    return summary

## Gradio

In [15]:
def transcribe_with_speakerlabels(audio_input, diarization_model, transcription_model, min_speakers, max_speakers):

    if audio_input is None:
        # Gradio allows raising gr.Error for user-friendly messages
        raise gr.Error("Please provide an audio input.")
    audio_filepath = None

    if isinstance(audio_input, str): # This means it's an uploaded file path
        print('audio_input is a filepath')
        audio_filepath = audio_input
    elif isinstance(audio_input, tuple): # This means it's microphone input (sample_rate, audio_data)
        sample_rate, audio_data = audio_input
        # Save the audio data to a temporary WAV file that pyannote and transformers can read
        temp_audio_file = "temp_gradio_mic_input.wav"
        sf.write(temp_audio_file, audio_data, sample_rate)
        audio_filepath = temp_audio_file
    else:
        raise gr.Error("Unsupported audio input format.")

    diarization = audio_diarization(audio_filepath, diarization_model, min_speakers, max_speakers)
    diarization_segments = extract_diarization_segments(diarization)

    transcription_result  = transcribe(audio_filepath, transcription_model)
    speaker_annotated_transcription = combine_diarization_transcription(diarization_segments, transcription_result)
    return speaker_annotated_transcription

def summarize(markdown_transcription, summarization_model_name, quantization):
    messages_payload = messages(markdown_transcription)
    summary = generate_summary(messages_payload, summarization_model_name, quantization)
    return summary

In [57]:
import gradio as gr

from gradio.components import label
# Build the Gradio interface
with gr.Blocks(theme=gr.themes.Soft(), title="Conversation Summarizer") as demo:
    gr.Markdown("""
    # Conversation summarizer Assistant

    **Features:** Multi-model support • Real-time streaming • Voice I/O • Speaker Diarization • Audio Transcription • Conversation Summarization
    """)

    with gr.Row():
        with gr.Column(scale = 2):
            gr.Markdown("### Transcription:")
            with gr.Group():
                transcription = gr.Markdown(
                    label="Transcription",
                    height = 800
                )
            with gr.Row():
                with gr.Column():
                    summarize_btn = gr.Button(
                        value = "📝 Summarize")
        with gr.Row():
            with gr.Column(scale =1):
                Audio = gr.Audio(
                    sources=["microphone", "upload"],
                    container = True,
                    type="filepath",
                    label="Upload or Record Audio")

            with gr.Column(scale =1):
                gr.Markdown("### ⚙️ Settings")

                diarization_model_choice = gr.Radio(
                    choices = [("speaker-diarization-3.1","pyannote/speaker-diarization-3.1"),("speaker-diarization-3.2","pyannote/speaker-diarization-3.2") ],
                    value  = "pyannote/speaker-diarization-3.1",
                    label  = "Audio Diarization Model",
                    info   = "Choose which diarization model to use"
                    )

                transcription_model_choice = gr.Radio(
                    choices = [("whisper-small","openai/whisper-small"),("whisper-medium","openai/whisper-medium.en")],
                    value  = "openai/whisper-medium.en",
                    label  = "Transcription Model",
                    info   = "Choose which transcription model to use"
                    )
                min_speakers = gr.Slider(
                    minimum=1,
                    maximum=10,
                    value=2,
                    step=1,
                )
                max_speakers = gr.Slider(
                    minimum=2,
                    maximum=10,
                    value=5,
                    step=1,
                )
                summarization_model_choice = gr.Radio(
                    choices = [("Llama-3.2-1B-Instruct","meta-llama/Llama-3.2-1B-Instruct"),
                               ("Llama-3.2-3B-Instruct","meta-llama/Llama-3.2-3B-Instruct"),
                                ("Llama-3.1-8B-Instruct","meta-llama/Llama-3.1-8B-Instruct")],
                    value =  "meta-llama/Llama-3.2-3B-Instruct",
                    label  = "Summarization Model",
                    info   = "Choose which summarization model to use"
                    )
                quantization = gr.Checkbox(
                    value = True,
                    label = "Quantization",
                    info = "Quantize the model for faster inference"
                )



    with gr.Row():
        with gr.Column():
            gr.Markdown("### Conversation summary:")
            with gr.Group():
                summary = gr.Markdown(
                    label="Conversation summary",
                    height=500
                    )
            with gr.Column():
                    summary_clear_btn = gr.ClearButton(
                    value = "🗑️ Clear summary",
                    scale  = 1,
                    variant="stop")
            clear_btn = gr.ClearButton(
                value = "🗑️ Clear all",
                scale   = 1,
                variant="stop")



    Audio.upload(
        fn = transcribe_with_speakerlabels,
        inputs = [Audio, diarization_model_choice, transcription_model_choice, min_speakers, max_speakers],
        outputs = [transcription]
    )

    summarize_btn.click(
        fn = summarize,
        inputs = [transcription, summarization_model_choice, quantization],
        outputs = [summary]
    )

    clear_btn.click(lambda: ["", ""], None, [transcription, summary])
    summary_clear_btn.click(lambda: "", None, summary)

    demo.launch(share=False, inbrowser=True, debug=True)

/tmp/ipykernel_1556/666432534.py:5: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Conversation Summarizer") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

Keyboard interruption in main thread... closing server.
